# GBD_11 — 2-D angular-spectrum reference + Era 2 boundary check

**The official Era 2 boundary check** (per CONCEPTS.md Topic 2's validation cadence). This is what GBD_7 + GBD_8 together did for 1-D, collapsed into one notebook because we already learned in Era 1 that doing them separately at toy parameters wasted a notebook.

**What it does:**
1. Implements an independent 2-D angular-spectrum (FFT) propagator — the gold standard for free-space scalar diffraction.
2. Builds a three-way field comparison: AS-of-source (the true answer), AS-of-GBDz0 (the answer for the field GBD started from), GBD propagation.
3. Sweeps z from 1 mm to 50 m and quantifies propagation-only and end-to-end L2 relative error.
4. Headline plot: log-scale L2 rel vs. z with the decomposition floor marked.

**If GBD agrees with AS to within the decomposition floor over the whole z sweep → Era 2 is closed.**

**Engineering trade-off vs. GBD_8:** GBD_8 used a ±500 mm × 10 000-pt grid in 1-D. The 2-D version of that would be 10000² = 100M complex points = 1.6 GB per field — too big. We compromise with ±100 mm × 513 pts (dx ≈ 390 µm). At z = 50 m, w(z) ≈ 24.7 mm, comfortably inside the grid (4σ coverage). **Important basis-design lesson learned mid-notebook (v1 → v2):** first attempt used w_basis = 0.4 mm (matching GBD_10), but on the new dx ≈ 390 µm grid that is only 1 sample per basis radius — severely undersampled. The result was a 1.4% prop-only L2 error plateau across all z (an aliasing artifact, not a method error). Switching to w_basis = 1 mm (matching the source waist, 2.56 samples per radius) cleared the artifact and gave textbook-clean linear-in-z paraxial-only error. Recorded in DISCUSSION_LOG Surprise 6.

**Predictions before running:**
- Decomposition floor at z = 0: probably 10⁻² to 10⁻¹ (worse than GBD_10 due to coarser dx).
- Propagation-only error: scales as `θ_div²·z/zR ≈ 1.2×10⁻⁷·z[m]`. Way below decomposition floor across the whole sweep.
- End-to-end error: stays at the decomposition floor.
- Possible wrap-around at z ≥ 30 m if the beam approaches the grid edge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# --- LiDAR parameters (carried forward from GBD_8/9/10) ---
wavelength = 1550e-9
w0_source  = 1.0e-3
w_basis    = 1.0e-3   # CHANGED v2: was 0.4mm; underampled at dx=390um (1 sample/radius). 1mm gives 2.56 samples/radius, eliminates aliasing artifact.
k          = 2*np.pi / wavelength
zR_source  = np.pi * w0_source**2 / wavelength
zR_basis   = np.pi * w_basis**2 / wavelength
theta_div_source = wavelength / (np.pi * w0_source)

print(f'wavelength  = {wavelength*1e9:.1f} nm')
print(f'w0_source   = {w0_source*1e3:.3f} mm')
print(f'w_basis     = {w_basis*1e3:.3f} mm')
print(f'zR_source   = {zR_source:.4f} m')
print(f'theta_div   = {theta_div_source:.3e} rad   (paraxial OK)')

# --- 2-D grid: wider than GBD_9/10, coarser dx, sized for z up to ~50 m ---
Lx = Ly = 100e-3              # ±100 mm
Nx = Ny = 513                 # odd → exact origin sample (CONCEPTS.md Topic 3 lesson)
x = np.linspace(-Lx, Lx, Nx)
y = np.linspace(-Ly, Ly, Ny)
X, Y = np.meshgrid(x, y, indexing='xy')
dx = x[1] - x[0]
print(f'\nGrid: {Nx}x{Ny}, x in [{-Lx*1e3:.0f}, {Lx*1e3:.0f}] mm')
print(f'dx = {dx*1e6:.1f} um  (basis radius w_basis = {w_basis*1e6:.0f} um → {w_basis/dx:.1f} samples per radius)')
print(f'At z = 50 m, w(z) = {w0_source*np.sqrt(1+(50/zR_source)**2)*1e3:.1f} mm  → grid extends to {Lx/(w0_source*np.sqrt(1+(50/zR_source)**2)):.1f}σ')

## The 2-D propagator (carried over from GBD_9/10)

Self-contained per project convention.

In [ ]:
def propagate_tilted_gaussian_2d(X, Y, x0, y0, w0, kx, ky, k, z):
    """GBD's analytic per-beamlet propagator — see CONCEPTS.md Topic 3."""
    wl = 2*np.pi / k
    zR_loc = np.pi * w0**2 / wl
    x_c = x0 + (kx/k) * z
    y_c = y0 + (ky/k) * z
    q0 = -1j * zR_loc
    qz = z + q0
    transverse = (q0/qz) * np.exp(1j * k * ((X-x_c)**2 + (Y-y_c)**2) / (2*qz))
    kz_long = np.sqrt(k**2 - kx**2 - ky**2)
    longitudinal = np.exp(1j * kz_long * z)
    tilt = np.exp(1j * (kx*X + ky*Y))
    return transverse * longitudinal * tilt

def propagate_basis_2d(X, Y, beam_info, w_basis, k, z, coefficients):
    """GBD basis sum — carried over from GBD_10."""
    Ny_loc, Nx_loc = X.shape
    Gp = np.zeros((Ny_loc*Nx_loc, len(beam_info)), dtype=complex)
    for n, (x0n, y0n, kxn, kyn) in enumerate(beam_info):
        Eb = propagate_tilted_gaussian_2d(X, Y, x0n, y0n, w_basis, kxn, kyn, k, z=z)
        Gp[:, n] = Eb.ravel()
    return (Gp @ coefficients).reshape(Ny_loc, Nx_loc)

## NEW: 2-D angular-spectrum (FFT) propagator

The independent reference. Three lines: forward FFT → multiply by transfer function `exp(i·kz·z)` → inverse FFT.

Uses complex `sqrt(k² − kx² − ky²)` so evanescent modes (those with `kx² + ky² > k²`) decay exponentially rather than oscillating, which is the physically-correct behavior.

In [ ]:
def propagate_angular_spectrum_2d(E0, x, y, k, z):
    """Independent 2-D AS propagator. Gold standard for free-space scalar diffraction."""
    Nx_loc, Ny_loc = len(x), len(y)
    dx_loc = x[1] - x[0]
    dy_loc = y[1] - y[0]
    kx_grid = 2*np.pi * np.fft.fftfreq(Nx_loc, d=dx_loc)
    ky_grid = 2*np.pi * np.fft.fftfreq(Ny_loc, d=dy_loc)
    KX, KY = np.meshgrid(kx_grid, ky_grid, indexing='xy')
    KZ = np.sqrt((k**2 - KX**2 - KY**2).astype(complex))   # complex sqrt → evanescent decay
    return np.fft.ifft2(np.fft.fft2(E0) * np.exp(1j * KZ * z))

# --- Sanity check 1: AS at z = 0 must be the identity ---
E_test = np.exp(-(X**2 + Y**2) / w0_source**2).astype(complex)
E_AS_z0 = propagate_angular_spectrum_2d(E_test, x, y, k, z=0.0)
print(f'[Sanity 1] Max |AS(E_test, z=0) - E_test| = {np.max(np.abs(E_AS_z0 - E_test)):.3e}')
print('(should be ~1e-15)')

## Build the basis (same as GBD_10) and solve LSQ on the new grid

9×9 = 81 beamlets at w_basis = 1 mm (matching source waist), spaced 1.5 mm, ±6 mm extent. **Note basis design changed from GBD_10's w=0.4mm/spacing=0.5mm** — the GBD_10 design was undersampled on this notebook's coarser grid. See markdown intro for the v1→v2 story.

In [ ]:
n_per_axis = 9
spacing    = 1.5e-3   # CHANGED v2: scaled with wider basis. 9x9 basis covers ±6mm.
x0_grid = (np.arange(n_per_axis) - (n_per_axis-1)/2) * spacing
y0_grid = (np.arange(n_per_axis) - (n_per_axis-1)/2) * spacing
beam_info = [(x0i, y0i, 0.0, 0.0) for x0i in x0_grid for y0i in y0_grid]
N_beams = len(beam_info)
print(f'N_beams = {N_beams}')

# --- source ---
E_source = np.exp(-(X**2 + Y**2) / w0_source**2).astype(complex)

# --- build G on the new grid ---
t0 = time.time()
G = np.zeros((Nx*Ny, N_beams), dtype=complex)
for n, (x0n, y0n, kxn, kyn) in enumerate(beam_info):
    Eb = propagate_tilted_gaussian_2d(X, Y, x0n, y0n, w_basis, kxn, kyn, k, z=0.0)
    G[:, n] = Eb.ravel()
print(f'Built G in {time.time()-t0:.2f} s, size {G.nbytes/1e6:.0f} MB')

# --- solve LSQ ---
lambda_reg = 1e-8
GhG = G.conj().T @ G
Ghe = G.conj().T @ E_source.ravel()
A = GhG + lambda_reg * np.eye(N_beams)
c = np.linalg.solve(A, Ghe)

# --- decomposition floor ---
E_recon_z0 = (G @ c).reshape(Ny, Nx)
L2_floor = np.linalg.norm(E_recon_z0 - E_source) / np.linalg.norm(E_source)
print(f'\n[Sanity 2] Decomposition L2 rel at z=0: {L2_floor:.3e}')
print(f'(GBD_10 had 9.93e-3 on a finer grid — coarser dx may push this up)')

## Three-way comparison at z = zR_source ≈ 2 m

Compute three fields and the two residuals.

In [ ]:
z_test = zR_source

# 1. AS of true source
t0 = time.time()
E_AS_source = propagate_angular_spectrum_2d(E_source, x, y, k, z_test)
t_AS = time.time() - t0

# 2. AS of GBD reconstruction at z = 0 (the field GBD actually started from)
E_AS_GBDz0 = propagate_angular_spectrum_2d(E_recon_z0, x, y, k, z_test)

# 3. GBD propagation
t0 = time.time()
E_GBD = propagate_basis_2d(X, Y, beam_info, w_basis, k, z=z_test, coefficients=c)
t_GBD = time.time() - t0

print(f'AS time:  {t_AS*1e3:.0f} ms')
print(f'GBD time: {t_GBD*1e3:.0f} ms')

# residuals
L2_prop = np.linalg.norm(E_GBD - E_AS_GBDz0) / np.linalg.norm(E_AS_GBDz0)
L2_e2e  = np.linalg.norm(E_GBD - E_AS_source) / np.linalg.norm(E_AS_source)
print(f'\n[Sanity 3] At z = zR_source = {z_test:.3f} m:')
print(f'  Propagation-only L2 rel = {L2_prop:.3e}   (expect ~1e-7 from theta_div^2 * z/zR)')
print(f'  End-to-end       L2 rel = {L2_e2e:.3e}   (expect ~ decomposition floor)')

## Z sweep over the LiDAR range

Log-spaced z from 1 mm to 50 m. At each z compute prop-only and end-to-end residuals. This is the headline measurement that closes Era 2.

In [ ]:
z_values = np.array([1e-3, 0.01, 0.1, 1.0, zR_source, 5.0, 10.0, 20.0, 50.0])

rows = []
t_total = time.time()
for z_v in z_values:
    E_AS_src = propagate_angular_spectrum_2d(E_source,    x, y, k, z_v)
    E_AS_g0  = propagate_angular_spectrum_2d(E_recon_z0,  x, y, k, z_v)
    E_g      = propagate_basis_2d(X, Y, beam_info, w_basis, k, z=z_v, coefficients=c)
    w_z = w0_source * np.sqrt(1 + (z_v/zR_source)**2)
    L2_p  = np.linalg.norm(E_g - E_AS_g0)  / np.linalg.norm(E_AS_g0)
    L2_e  = np.linalg.norm(E_g - E_AS_src) / np.linalg.norm(E_AS_src)
    rows.append((z_v, w_z, L2_p, L2_e))
print(f'Z sweep finished in {time.time()-t_total:.1f} s\n')

print(f"{'z [m]':>8s}  {'w(z) [mm]':>10s}  {'prop-only':>12s}  {'end-to-end':>12s}")
for z_v, w_z, L2_p, L2_e in rows:
    print(f'{z_v:8.4f}  {w_z*1e3:10.3f}  {L2_p:12.3e}  {L2_e:12.3e}')

## Headline plot — log-scale L2 rel vs. z

Both residuals on log scale. Decomposition floor as a horizontal dashed line. The signature we expect: end-to-end pinned at the floor, prop-only growing slowly with z (paraxial leading correction).

In [ ]:
z_arr = np.array([r[0] for r in rows])
L2_p_arr = np.array([r[2] for r in rows])
L2_e_arr = np.array([r[3] for r in rows])

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(z_arr, L2_p_arr, 'o-', label='Propagation-only  |E_GBD − AS(GBDz0)|')
ax.loglog(z_arr, L2_e_arr, 's-', label='End-to-end        |E_GBD − AS(source)|')
ax.axhline(L2_floor, ls='--', color='k', alpha=0.5, label=f'Decomposition floor ({L2_floor:.2e})')
# theoretical paraxial correction line
z_theory = np.logspace(-3, 2, 100)
L2_theory = theta_div_source**2 * z_theory / zR_source
ax.loglog(z_theory, L2_theory, ':', color='gray', alpha=0.7, label=r'$\theta_{div}^2 \cdot z/z_R$  (theory)')
ax.set_xlabel('z [m]')
ax.set_ylabel('L2 relative error')
ax.set_title('GBD vs. AS at LiDAR-realistic 2-D parameters')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Visual confirmation at one z

Side-by-side amplitude and phase panels at z = zR_source: AS-of-source, GBD, and their difference.

In [ ]:
z_v = zR_source
E_AS_src = propagate_angular_spectrum_2d(E_source, x, y, k, z_v)
E_g      = propagate_basis_2d(X, Y, beam_info, w_basis, k, z=z_v, coefficients=c)
kz_loc = k
extent = [-Lx*1e3, Lx*1e3, -Ly*1e3, Ly*1e3]

# only plot the central ±10mm region (the beam is localised there)
ix_min = np.argmin(np.abs(x + 10e-3)); ix_max = np.argmin(np.abs(x - 10e-3))
iy_min = np.argmin(np.abs(y + 10e-3)); iy_max = np.argmin(np.abs(y - 10e-3))
extent_z = [x[ix_min]*1e3, x[ix_max]*1e3, y[iy_min]*1e3, y[iy_max]*1e3]
def crop(E):
    return E[iy_min:iy_max, ix_min:ix_max]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, (label, E) in enumerate([('AS(source)', E_AS_src), ('GBD', E_g), ('GBD − AS', E_g - E_AS_src)]):
    ax = axes[0, col]
    im = ax.imshow(np.abs(crop(E)), origin='lower', extent=extent_z, cmap='viridis')
    ax.set_title(f'|{label}|, z = {z_v:.3f} m')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax = axes[1, col]
    E_env = crop(E) * np.exp(-1j*kz_loc*z_v)
    im = ax.imshow(np.angle(E_env), origin='lower', extent=extent_z, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    ax.set_title(f'arg({label})')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Summary — what this notebook validates

If the headline plot shows:
- End-to-end L2 rel pinned at the decomposition floor across all z, and
- Prop-only L2 rel growing as `θ_div²·z/zR` (the textbook paraxial correction), well below the floor,

then **Era 2 is closed**. GBD has been validated in 2-D at LiDAR-realistic parameters across the LiDAR operating range. The propagation engine introduces zero error beyond the decomposition floor; everything we see is basis-design error.

**What this notebook deliberately did NOT do** (saving for later):
- Push z beyond 50 m — would need a wider grid (more memory) or accept wrap-around. Defer to a future notebook if needed.
- Add transmit optics, scanner, atmosphere, target — all Era 3.
- Stress-test on non-Gaussian sources — GBD_12 (only if needed).
- GPU acceleration / vectorization — premature.

**Era 2 status after this notebook:** if all checks pass, Era 2 is complete. GBD_12 becomes optional. Next major step: Era 3 — transmit optics, scanner, atmospheric propagation.